In [1]:
# Import required libraries and set up our environment
import pandas as pd
from pprint import PrettyPrinter

import boto3
import time


In [2]:

print("📚 Setting up the environment...")
pp = PrettyPrinter(indent=2)
translate = boto3.client("translate")
print("✅ Environment setup complete!")
print(f"🌍 Using AWS region: {translate.meta.region_name}")
print("✅ AWS Translate client ready")

📚 Setting up the environment...
✅ Environment setup complete!
🌍 Using AWS region: eu-west-1
✅ AWS Translate client ready


In [9]:
# Load the dataset

print("📄 Loading articles.csv...")

df = pd.read_csv("articles.csv")

print("✅ Dataset loaded")
print("Columns:", list(df.columns))
print("Language counts:")
print(df["language"].value_counts())

📄 Loading articles.csv...
✅ Dataset loaded
Columns: ['source', 'language', 'url', 'article_text']
Language counts:
language
en    18
bn    12
Name: count, dtype: int64


In [10]:
df

,source,language,url,article_text
0,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The interim government's revised election time...
1,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,BNP has announced its candidates for 36 more c...
2,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,Discontent runs deep among BNP's partners as t...
3,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,Even though a week has passed since the announ...
4,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The next national election will be tough for t...
5,The Daily Star,en,https://www.thedailystar.net/opinion/views/new...,"An opinion piece titled ""BNP's notes of dissen..."
6,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The BNP has survived Sheikh Hasina's 15-year r...
7,The Daily Star,en,https://www.thedailystar.net/opinion/views/new...,The former Prime Minister Khaleda Zia on Tuesd...
8,The Daily Star,en,https://www.thedailystar.net/opinion/editorial...,After having weathered a difficult 15 years in...
9,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The BNP has alerted its election candidates th...


In [5]:
def translate_to_en(text, source_lang):
    """
    Translate text to English using Amazon Translate.
    Safely handles empty or NaN values.
    """
    if text is None or (isinstance(text, float) and pd.isna(text)) or str(text).strip() == "":
        return None

    response = translate.translate_text(
        Text=str(text)[:5000],            # AWS limit
        SourceLanguageCode=source_lang,
        TargetLanguageCode="en"
    )

    return response["TranslatedText"]


In [6]:
def translate_non_english_rows(
    df,
    text_col="article_text",
    lang_col="language",
    sleep_s=0.1
):
    df = df.copy()

    # Ensure bookkeeping columns exist
    if "text_orig_ln" not in df.columns:
        df["text_orig_ln"] = pd.NA
    if "translated_text" not in df.columns:
        df["translated_text"] = pd.NA
    if "translate_error" not in df.columns:
        df["translate_error"] = pd.NA

    # Mask for non-English rows
    mask = (
        df[lang_col].notna()
        & (df[lang_col].str.lower() != "en")
        & df[text_col].notna()
    )

    # Freeze original non-English text
    df.loc[mask, "text_orig_ln"] = df.loc[mask, text_col]

    print("🔁 Translating non-English articles...")

    for idx, row in df.loc[mask, [text_col, lang_col]].iterrows():
        try:
            df.at[idx, "translated_text"] = translate_to_en(
                row[text_col],
                row[lang_col].lower()
            )
            print(f"✅ Translated row {idx}")

        except Exception as e:
            df.at[idx, "translate_error"] = str(e)
            print(f"❌ Translation failed at row {idx}: {str(e)}")

        if sleep_s:
            time.sleep(sleep_s)

    # English rows → copy text directly
    en_mask = df[lang_col].str.lower() == "en"
    df.loc[en_mask, "translated_text"] = df.loc[en_mask, text_col]

    return df

In [7]:
translated_df = translate_non_english_rows(
    df,
    text_col="article_text",
    lang_col="language",
    sleep_s=0.1
)

🔁 Translating non-English articles...
✅ Translated row 10
✅ Translated row 11
✅ Translated row 12
✅ Translated row 13
✅ Translated row 14
✅ Translated row 15
✅ Translated row 16
✅ Translated row 17
✅ Translated row 18
✅ Translated row 21
✅ Translated row 24
✅ Translated row 26


In [14]:
translated_df

,source,language,url,article_text,text_orig_ln,translated_text,translate_error
0,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The interim government's revised election time...,<NA>,The interim government's revised election time...,<NA>
1,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,BNP has announced its candidates for 36 more c...,<NA>,BNP has announced its candidates for 36 more c...,<NA>
2,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,Discontent runs deep among BNP's partners as t...,<NA>,Discontent runs deep among BNP's partners as t...,<NA>
3,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,Even though a week has passed since the announ...,<NA>,Even though a week has passed since the announ...,<NA>
4,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The next national election will be tough for t...,<NA>,The next national election will be tough for t...,<NA>
5,The Daily Star,en,https://www.thedailystar.net/opinion/views/new...,"An opinion piece titled ""BNP's notes of dissen...",<NA>,"An opinion piece titled ""BNP's notes of dissen...",<NA>
6,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The BNP has survived Sheikh Hasina's 15-year r...,<NA>,The BNP has survived Sheikh Hasina's 15-year r...,<NA>
7,The Daily Star,en,https://www.thedailystar.net/opinion/views/new...,The former Prime Minister Khaleda Zia on Tuesd...,<NA>,The former Prime Minister Khaleda Zia on Tuesd...,<NA>
8,The Daily Star,en,https://www.thedailystar.net/opinion/editorial...,After having weathered a difficult 15 years in...,<NA>,After having weathered a difficult 15 years in...,<NA>
9,The Daily Star,en,https://www.thedailystar.net/news/bangladesh/p...,The BNP has alerted its election candidates th...,<NA>,The BNP has alerted its election candidates th...,<NA>


In [12]:
import pandas as pd
import sys

# Assuming you have your dataframe loaded
df = pd.read_csv("articles.csv")

def translate_logic(row):
    text = row['article_text']
    lang = row['language']
    
    # 1. If English, return as is
    if lang == "en":
        return text
    
    # 2. If Bengali, check size for translation
    if lang == "bn":
        byte_size = len(text.encode('utf-8'))
        
        if byte_size <= 10000:
            # Small enough for a single call
            resp = translate.translate_text(Text=text, SourceLanguageCode="bn", TargetLanguageCode="en")
            return resp['TranslatedText']
        
        else:
            # Logic for long articles: Split into chunks
            chunk_size = 3000
            chunks = [
                text[0:chunk_size],
                text[chunk_size:chunk_size*2],
                text[chunk_size*2:chunk_size*3],
                text[chunk_size*3:]
            ]
            
            translated_parts = []
            for chunk in chunks:
                if chunk.strip():
                    resp = translate.translate_text(
                        Text=chunk, 
                        SourceLanguageCode="bn", 
                        TargetLanguageCode="en"
                    )
                    translated_parts.append(resp['TranslatedText'])
            
            return "\n".join(translated_parts)
            
    return text  # Fallback for any other languages

# Apply the function to create the new column
print("🔄 Translating articles...")
df['translated_text'] = df.apply(translate_logic, axis=1)

# Save back to CSV
df.to_csv("articles_updated.csv", index=False)
print("✅ Translation complete and saved!")

🔄 Translating articles...
✅ Translation complete and saved!


In [13]:
len(df["translated_text"])

30

In [16]:
article_list = pd.read_csv("article_list.csv")
article_list["translated_text"] = df["translated_text"]

In [ ]:
article_list.iloc

,Channels,News Title,News url,Language,Sentiment,translated_text
0,The Daily Star,New Polls Timing BNP upbeat process irks Jamaa...,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The interim government's revised election time...
1,The Daily Star,BNP announces candidates for 36 more constitue...,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,BNP has announced its candidates for 36 more c...
2,The Daily Star,Seat-sharing snub riles BNP allies,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,Discontent runs deep among BNP's partners as t...
3,The Daily Star,Seat sharing BNP still keeps allies hanging,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,Even though a week has passed since the announ...
4,The Daily Star,"Election to be tough for BNP, bigger test if v...",https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The next national election will be tough for t...
5,The Daily Star,Old habits die hard? BNP’s response,https://www.thedailystar.net/opinion/views/new...,en,NaN,"An opinion piece titled ""BNP's notes of dissen..."
6,The Daily Star,BNP at 47: Caught between prospects and perils,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The BNP has survived Sheikh Hasina's 15-year r...
7,The Daily Star,BNP’s show of force on Airport Road and a poli...,https://www.thedailystar.net/opinion/views/new...,en,NaN,The former Prime Minister Khaleda Zia on Tuesd...
8,The Daily Star,Fulfilling Uprising's Aspirations: Correction ...,https://www.thedailystar.net/opinion/editorial...,en,NaN,After having weathered a difficult 15 years in...
9,Prothom Alo,BNP rethinks its approach to allies,https://en.prothomalo.com/bangladesh/politics/...,en,NaN,The BNP has alerted its election candidates th...
